In [5]:
import os
import time
start_notebook = time.time()
os.environ["JAVA_HOME"] = r"C:\java\openjdk-17.0.18b8"
os.environ["HADOOP_HOME"] = "C:\\java\\hadoop-3.4.3"
os.environ["SPARK_HOME"] = r"C:\java\spark-4.1.1-bin-hadoop3"
os.environ["HADOOP_CONF_DIR"] = r"C:\java\hadoop-3.4.3\etc\hadoop"
os.environ["YARN_CONF_DIR"] = r"C:\java\hadoop-3.4.3\etc\hadoop"

In [6]:
from pyspark.sql import SparkSession
import sys

spark = SparkSession.builder \
    .appName("FlightDelayPredictionBigData") \
    .master("yarn") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://master:9000") \
    .config("spark.driver.memory", "3g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.memoryOverhead", "512m") \
    .config("spark.executor.instances", "2") \
    .config("spark.executor.cores", "4") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "false") \
    .config("spark.pyspark.python", sys.executable) \
    .config("spark.pyspark.driver.python", sys.executable) \
    .config("spark.hadoop.yarn.resourcemanager.scheduler.address", "master:8030") \
    .config("spark.hadoop.yarn.resourcemanager.address", "master:8032") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.yarn.am.waitTime", "800s") \
    .config("spark.rpc.askTimeout", "800s") \
    .config("spark.driver.host", "100.125.135.6") \
    .config("spark.yarn.jars", "local:///C:/java/spark-4.1.1-bin-hadoop3/jars/*") \
    .config("spark.yarn.dist.archives", "") \
    .config("spark.driver.host", "100.125.135.6") \
    .config("spark.driver.bindAddress", "100.125.135.6") \
    .getOrCreate()

In [7]:
HDFS_PATH = "hdfs://master:9000/data/"

In [8]:
airlines = spark.read.csv(HDFS_PATH + "airlines.csv", header=True, inferSchema=True, sep = ",")
airlines.show(2)

+---------+--------------------+
|IATA_CODE|             AIRLINE|
+---------+--------------------+
|       UA|United Air Lines ...|
|       AA|American Airlines...|
+---------+--------------------+
only showing top 2 rows


In [9]:
airports = spark.read.csv(HDFS_PATH + "airports.csv", header=True, inferSchema=True, sep = ",")
airports.show(2)

+---------+--------------------+---------+-----+-------+--------+---------+
|IATA_CODE|             AIRPORT|     CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+--------------------+---------+-----+-------+--------+---------+
|      ABE|Lehigh Valley Int...|Allentown|   PA|    USA|40.65236| -75.4404|
|      ABI|Abilene Regional ...|  Abilene|   TX|    USA|32.41132| -99.6819|
+---------+--------------------+---------+-----+-------+--------+---------+
only showing top 2 rows


In [10]:
flights = spark.read.csv(HDFS_PATH + "flights.csv", header=True, inferSchema=True, sep = ",")
flights.show(2)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

In [11]:
flights = flights.repartition(16)

In [ ]:
from pyspark.sql.functions import col, when, lpad, concat, lit, substring

TIME_COLS = ['SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME']

for tcol in TIME_COLS:
    time_string_col = lpad(
        when(col(tcol) == 2400, 0)
        .otherwise(col(tcol))
        .cast("int").cast("string"),
        4, "0"
    )

    flights = flights.withColumn(
        tcol,
        concat(substring(time_string_col, 1, 2), lit(":"), substring(time_string_col, 3, 2))
    )

In [13]:
from pyspark.sql import functions as F

flights = flights.filter(F.col("CANCELLED") == 0)

flights = flights.withColumn(
    "label",
    F.when(F.col("ARRIVAL_DELAY") > 15, 1).otherwise(0)
)

In [14]:
flights = flights.withColumn(
    "DEP_TIME_MINUTES",
    F.split(F.col("DEPARTURE_TIME"), ":")[0].cast("int") * 60 +
    F.split(F.col("DEPARTURE_TIME"), ":")[1].cast("int")
)

In [15]:
flights.show(2)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+-----+----------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|label|DEP_TIME_MINUTES|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+---

In [16]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DEP_TIME_MINUTES", "DISTANCE", "DEPARTURE_DELAY"]

stages = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages.append(assembler)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
    # maxIter=20,
    # regParam=0.1,
    # elasticNetParam=0.0
)
stages.append(lr)

pipeline = Pipeline(stages=stages)

In [17]:
train_data, test_data = flights.randomSplit([0.8, 0.2], seed=42)

total_count = train_data.count()
delay_count = train_data.filter(F.col("label") == 1.0).count()
on_time_count = total_count - delay_count

weight_for_delay = total_count / (2.0 * delay_count)
weight_for_ontime = total_count / (2.0 * on_time_count)

train_data_weighted = train_data.withColumn(
    "classWeight",
    F.when(F.col("label") == 1.0, weight_for_delay).otherwise(weight_for_ontime)
)

# ── OPTIMIZATION: Cache & Persist ──────────────────────────────
print("Caching flights dataset...")
flights.cache()

print("Persisting train/test splits...")
train_data.persist()
test_data.persist()
train_data_weighted.persist()

# print(f"Train size: {train_data.count():,}")
# print(f"Test size:  {test_data.count():,}")
print("Cache & Persist completed.")

model = pipeline.fit(train_data_weighted)

predictions = model.transform(test_data)

Caching flights dataset...
Persisting train/test splits...
Cache & Persist completed.


In [18]:
model_path = "model/flight_logistic_model"
model.write().overwrite().save(model_path)

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9363
2. Accuracy:      0.9073
3. Precision:     0.9154
4. Recall:        0.9073
5. F1-Score:      0.9102


In [20]:
from sklearn.metrics import classification_report
y_compare = predictions.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare["label"], y_compare["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.92      0.94    940259
       Delay       0.70      0.83      0.76    204619

    accuracy                           0.91   1144878
   macro avg       0.83      0.88      0.85   1144878
weighted avg       0.92      0.91      0.91   1144878



In [21]:
import pandas as pd
import numpy as np

lr_model = model.stages[-1]
coefficients = lr_model.coefficients.toArray()

features_metadata = predictions.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

feature_imp_df = feature_imp_df.sort_values(by='Absolute_Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===
                     Feature_Name  Coefficient  Absolute_Importance
0   DESTINATION_AIRPORT_Vec_11503   -10.639539            10.639539
1   DESTINATION_AIRPORT_Vec_13459   -10.418537            10.418537
2   DESTINATION_AIRPORT_Vec_13541    -9.936735             9.936735
3   DESTINATION_AIRPORT_Vec_10666    -9.649258             9.649258
4   DESTINATION_AIRPORT_Vec_12016    -9.202045             9.202045
5   DESTINATION_AIRPORT_Vec_13127    -9.143759             9.143759
6        ORIGIN_AIRPORT_Vec_10268    -8.927787             8.927787
7   DESTINATION_AIRPORT_Vec_14222    -8.871042             8.871042
8   DESTINATION_AIRPORT_Vec_10165    -8.623587             8.623587
9        ORIGIN_AIRPORT_Vec_13502    -8.531564             8.531564
10       ORIGIN_AIRPORT_Vec_11503    -8.365854             8.365854
11       ORIGIN_AIRPORT_Vec_12343    -7.295387             7.295387
12  DESTINATION_AIRPORT_Vec_11525    -7.215774         

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DEP_TIME_MINUTES", "DISTANCE", "DEPARTURE_DELAY"]

stages_dt = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_dt += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_dt = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_dt.append(assembler_dt)

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",   # thêm
    maxDepth=5,
    maxBins=32,
    impurity="gini"
)
stages_dt.append(dt)

pipeline_dt = Pipeline(stages=stages_dt)

In [23]:
model_dt = pipeline_dt.fit(train_data_weighted)

predictions_dt = model_dt.transform(test_data)

In [24]:
model_dt_path = "model/flight_decision_tree_model"
model_dt.write().overwrite().save(model_dt_path)

In [25]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_dt)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_dt)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_dt)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_dt)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_dt)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.7866
2. Accuracy:      0.9156
3. Precision:     0.9193
4. Recall:        0.9156
5. F1-Score:      0.9171


In [26]:
y_compare_dt = predictions_dt.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_dt["label"], y_compare_dt["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.94      0.95    940259
       Delay       0.74      0.81      0.77    204619

    accuracy                           0.92   1144878
   macro avg       0.85      0.87      0.86   1144878
weighted avg       0.92      0.92      0.92   1144878



In [27]:
dt_model = model_dt.stages[-1]
feature_importances = dt_model.featureImportances.toArray()

features_metadata = predictions_dt.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (DECISION TREE) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (DECISION TREE) ===
                     Feature_Name  Importance
0                 DEPARTURE_DELAY    0.998969
1                  AIRLINE_Vec_WN    0.000594
2                  AIRLINE_Vec_UA    0.000343
3     DESTINATION_AIRPORT_Vec_LAX    0.000093
4                     MONTH_Vec_3    0.000000
5                     MONTH_Vec_5    0.000000
6                    MONTH_Vec_10    0.000000
7                     MONTH_Vec_4    0.000000
8                    MONTH_Vec_12    0.000000
9                    MONTH_Vec_11    0.000000
10                    MONTH_Vec_9    0.000000
11                    MONTH_Vec_1    0.000000
12                    MONTH_Vec_2    0.000000
13    DESTINATION_AIRPORT_Vec_ITH    0.000000
14  DESTINATION_AIRPORT_Vec_12265    0.000000


### Phần phía dưới chưa chạy (chỉ test phần thời gian bên trên)

In [28]:
from pyspark.ml.classification import GBTClassifier

stages_gbt = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_gbt += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_gbt = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_gbt.append(assembler_gbt)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",   # thêm
    maxIter=10,
    maxDepth=4,
    stepSize=0.1
)
stages_gbt.append(gbt)

pipeline_gbt = Pipeline(stages=stages_gbt)

In [29]:
model_gbt = pipeline_gbt.fit(train_data_weighted)

predictions_gbt = model_gbt.transform(test_data)

In [30]:
model_gbt_path = "model/flight_gbt_model"
model_gbt.write().overwrite().save(model_gbt_path)

In [31]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_gbt)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_gbt)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_gbt)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_gbt)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_gbt)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9325
2. Accuracy:      0.9028
3. Precision:     0.9130
4. Recall:        0.9028
5. F1-Score:      0.9063


In [32]:
y_compare_gbt = predictions_gbt.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_gbt["label"], y_compare_gbt["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.92      0.94    940259
       Delay       0.69      0.83      0.75    204619

    accuracy                           0.90   1144878
   macro avg       0.83      0.88      0.85   1144878
weighted avg       0.91      0.90      0.91   1144878



In [33]:
gbt_model = model_gbt.stages[-1]
feature_importances = gbt_model.featureImportances.toArray()

features_metadata = predictions_gbt.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (GBT) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (GBT) ===
                   Feature_Name  Importance
0               DEPARTURE_DELAY    0.943372
1                AIRLINE_Vec_DL    0.014739
2                      DISTANCE    0.011508
3                AIRLINE_Vec_WN    0.011425
4                AIRLINE_Vec_UA    0.009170
5   DESTINATION_AIRPORT_Vec_LAX    0.003737
6        ORIGIN_AIRPORT_Vec_ATL    0.001276
7                AIRLINE_Vec_US    0.001274
8                AIRLINE_Vec_HA    0.001005
9   DESTINATION_AIRPORT_Vec_ORD    0.000880
10             DEP_TIME_MINUTES    0.000587
11               AIRLINE_Vec_EV    0.000458
12       ORIGIN_AIRPORT_Vec_LGA    0.000440
13            DAY_OF_WEEK_Vec_4    0.000088
14               AIRLINE_Vec_NK    0.000028


In [ ]:
total = train_data.count()
n_delay = train_data.filter(F.col("label") == 1).count()
n_ontime = train_data.filter(F.col("label") == 0).count()

weight_delay  = total / (2 * n_delay)
weight_ontime = total / (2 * n_ontime)

train_data_w = train_data.withColumn(
    "classWeight",
    F.when(F.col("label") == 1, weight_delay).otherwise(weight_ontime)
)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

stages_rf = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_rf += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_rf = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_rf.append(assembler_rf)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    numTrees=100,
    maxDepth=5,
    seed=42
)
stages_rf.append(rf)

pipeline_rf = Pipeline(stages=stages_rf)

In [ ]:
model_rf = pipeline_rf.fit(train_data_weighted)  # use train_data_w instead of train_data
predictions_rf = model_rf.transform(test_data)

In [37]:
model_rf_path = "model/flight_random_forest_model"
model_rf.write().overwrite().save(model_rf_path)

In [38]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_rf)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_rf)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_rf)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_rf)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_rf)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.8906
2. Accuracy:      0.9121
3. Precision:     0.9169
4. Recall:        0.9121
5. F1-Score:      0.9139


In [39]:
y_compare_rf = predictions_rf.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_rf["label"], y_compare_rf["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.93      0.95    940259
       Delay       0.73      0.81      0.77    204619

    accuracy                           0.91   1144878
   macro avg       0.84      0.87      0.86   1144878
weighted avg       0.92      0.91      0.91   1144878



In [40]:
rf_model = model_rf.stages[-1]
feature_importances = rf_model.featureImportances.toArray()

features_metadata = predictions_rf.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (RANDOM FOREST) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (RANDOM FOREST) ===
                   Feature_Name  Importance
0               DEPARTURE_DELAY    0.218771
1              DEP_TIME_MINUTES    0.152737
2                AIRLINE_Vec_NK    0.055276
3                AIRLINE_Vec_DL    0.053143
4                   MONTH_Vec_6    0.041681
5                  MONTH_Vec_10    0.027979
6             DAY_OF_WEEK_Vec_6    0.026135
7                   MONTH_Vec_9    0.025398
8   DESTINATION_AIRPORT_Vec_SLC    0.018090
9                AIRLINE_Vec_F9    0.017910
10               AIRLINE_Vec_HA    0.017843
11  DESTINATION_AIRPORT_Vec_LGA    0.017034
12               AIRLINE_Vec_AS    0.016724
13                  MONTH_Vec_2    0.015847
14       ORIGIN_AIRPORT_Vec_ORD    0.015381


In [41]:
from pyspark.ml.classification import LinearSVC

stages_svm = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_svm += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_svm = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_svm.append(assembler_svm)

svm = LinearSVC(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=20,
    regParam=0.1
)
stages_svm.append(svm)

pipeline_svm = Pipeline(stages=stages_svm)

In [42]:
model_svm = pipeline_svm.fit(train_data_weighted)
predictions_svm = model_svm.transform(test_data)

In [43]:
model_svm_path = "model/flight_svm_model"
model_svm.write().overwrite().save(model_svm_path)

In [44]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_svm)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_svm)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_svm)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_svm)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_svm)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9250
2. Accuracy:      0.9283
3. Precision:     0.9276
4. Recall:        0.9283
5. F1-Score:      0.9235


In [45]:
y_compare_svm = predictions_svm.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_svm["label"], y_compare_svm["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.93      0.99      0.96    940259
       Delay       0.92      0.66      0.77    204619

    accuracy                           0.93   1144878
   macro avg       0.92      0.82      0.86   1144878
weighted avg       0.93      0.93      0.92   1144878



In [46]:
svm_model = model_svm.stages[-1]
coefficients = svm_model.coefficients.toArray()

features_metadata = predictions_svm.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

feature_imp_df = feature_imp_df.sort_values(by='Absolute_Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (LINEAR SVM) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (LINEAR SVM) ===
                     Feature_Name  Coefficient  Absolute_Importance
0        ORIGIN_AIRPORT_Vec_13964     1.153455             1.153455
1          ORIGIN_AIRPORT_Vec_ADK     1.118869             1.118869
2   DESTINATION_AIRPORT_Vec_13964     1.063349             1.063349
3        ORIGIN_AIRPORT_Vec_13541     1.055799             1.055799
4        ORIGIN_AIRPORT_Vec_15497     0.996539             0.996539
5        ORIGIN_AIRPORT_Vec_14222     0.991165             0.991165
6        ORIGIN_AIRPORT_Vec_10165     0.979553             0.979553
7          ORIGIN_AIRPORT_Vec_GST     0.965554             0.965554
8   DESTINATION_AIRPORT_Vec_12016    -0.805266             0.805266
9   DESTINATION_AIRPORT_Vec_14711     0.802092             0.802092
10         ORIGIN_AIRPORT_Vec_PPG     0.790595             0.790595
11       ORIGIN_AIRPORT_Vec_14487    -0.722858             0.722858
12       ORIGIN_AIRPORT_Vec_10154     0.695832             0.695

In [47]:
from pyspark.ml.classification import NaiveBayes

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DISTANCE", "DEPARTURE_DELAY"]  # bỏ DEP_TIME_MINUTES vì có thể âm

stages_nb = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    stages_nb.append(indexer)

assembler_inputs = [col + "_Index" for col in categorical_cols] + numeric_cols
assembler_nb = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_nb.append(assembler_nb)

nb = NaiveBayes(
    featuresCol="features",
    labelCol="label",
    smoothing=1.0,
    modelType="gaussian"  # dùng gaussian vì features là continuous
)
stages_nb.append(nb)

pipeline_nb = Pipeline(stages=stages_nb)

In [48]:
model_nb = pipeline_nb.fit(train_data)
predictions_nb = model_nb.transform(test_data)

In [49]:
model_nb_path = "model/flight_naive_bayes_model"
model_nb.write().overwrite().save(model_nb_path)

In [50]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_nb)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_nb)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_nb)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_nb)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_nb)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.5278
2. Accuracy:      0.9333
3. Precision:     0.9315
4. Recall:        0.9333
5. F1-Score:      0.9303


In [51]:
y_compare_nb = predictions_nb.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_nb["label"], y_compare_nb["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.94      0.98      0.96    940259
       Delay       0.89      0.72      0.79    204619

    accuracy                           0.93   1144878
   macro avg       0.92      0.85      0.88   1144878
weighted avg       0.93      0.93      0.93   1144878



In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print("=== TUNING LOGISTIC REGRESSION ===")

stages_lr_tune = [s for s in stages if not isinstance(s, LogisticRegression)]

lr_tune = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
)
stages_lr_tune.append(lr_tune)
pipeline_lr_tune = Pipeline(stages=stages_lr_tune)

param_grid_lr = ParamGridBuilder() \
    .addGrid(lr_tune.maxIter, [20, 50]) \
    .addGrid(lr_tune.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr_tune.elasticNetParam, [0.0, 0.5]) \
    .build()

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

cv_lr = CrossValidator(
    estimator=pipeline_lr_tune,
    estimatorParamMaps=param_grid_lr,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

# Use 30% train_data_weighted for tuning
train_sample, _ = train_data_weighted.randomSplit([0.3, 0.7], seed=42)
print(f"Full train size:   {train_data_weighted.count()}")
print(f"Sample train size: {train_sample.count()}")

cv_model_lr = cv_lr.fit(train_sample)
best_model_lr = cv_model_lr.bestModel

print(f"Best maxIter:        {best_model_lr.stages[-1].getMaxIter()}")
print(f"Best regParam:       {best_model_lr.stages[-1].getRegParam()}")
print(f"Best elasticNet:     {best_model_lr.stages[-1].getElasticNetParam()}")

predictions_lr_tuned = best_model_lr.transform(test_data)

roc_auc = evaluator.evaluate(predictions_lr_tuned)
print(f"\nROC-AUC after tuning: {roc_auc:.4f}")

=== TUNING LOGISTIC REGRESSION ===
Full train size:   4584317
Sample train size: 1376098
Best maxIter:        50
Best regParam:       0.01
Best elasticNet:     0.5

ROC-AUC after tuning: 0.9304


In [53]:
print("=== RESULTS AFTER TUNING (LR) ===")

roc_auc = evaluator.evaluate(predictions_lr_tuned)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
print(f"2. Accuracy:      {acc_evaluator.evaluate(predictions_lr_tuned):.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
print(f"3. Precision:     {prec_evaluator.evaluate(predictions_lr_tuned):.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
print(f"4. Recall:        {rec_evaluator.evaluate(predictions_lr_tuned):.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
print(f"5. F1-Score:      {f1_evaluator.evaluate(predictions_lr_tuned):.4f}")

=== RESULTS AFTER TUNING (LR) ===
1. ROC-AUC Score: 0.9304
2. Accuracy:      0.9222
3. Precision:     0.9231
4. Recall:        0.9222
5. F1-Score:      0.9226


In [54]:
print("=== TUNING GBT ===")

stages_gbt_tune = [s for s in stages_gbt if not isinstance(s, GBTClassifier)]

gbt_tune = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
)
stages_gbt_tune.append(gbt_tune)
pipeline_gbt_tune = Pipeline(stages=stages_gbt_tune)

train_sample, _ = train_data_weighted.randomSplit([0.1, 0.9], seed=42)

param_grid_gbt = ParamGridBuilder() \
    .addGrid(gbt_tune.maxIter, [10, 20]) \
    .addGrid(gbt_tune.maxDepth, [4, 6]) \
    .build()

cv_gbt = CrossValidator(
    estimator=pipeline_gbt_tune,
    estimatorParamMaps=param_grid_gbt,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

cv_model_gbt = cv_gbt.fit(train_sample)
best_model_gbt = cv_model_gbt.bestModel

print(f"Best maxIter:    {best_model_gbt.stages[-1].getMaxIter()}")
print(f"Best maxDepth:   {best_model_gbt.stages[-1].getMaxDepth()}")

predictions_gbt_tuned = best_model_gbt.transform(test_data)

roc_auc = evaluator.evaluate(predictions_gbt_tuned)
print(f"\nROC-AUC after tuning: {roc_auc:.4f}")

=== TUNING GBT ===
Best maxIter:    20
Best maxDepth:   6

ROC-AUC after tuning: 0.9352


In [55]:
print("=== RESULTS AFTER TUNING (GBT) ===")

roc_auc = evaluator.evaluate(predictions_gbt_tuned)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

print(f"2. Accuracy:      {acc_evaluator.evaluate(predictions_gbt_tuned):.4f}")
print(f"3. Precision:     {prec_evaluator.evaluate(predictions_gbt_tuned):.4f}")
print(f"4. Recall:        {rec_evaluator.evaluate(predictions_gbt_tuned):.4f}")
print(f"5. F1-Score:      {f1_evaluator.evaluate(predictions_gbt_tuned):.4f}")

=== RESULTS AFTER TUNING (GBT) ===
1. ROC-AUC Score: 0.9352
2. Accuracy:      0.9126
3. Precision:     0.9179
4. Recall:        0.9126
5. F1-Score:      0.9146


In [56]:
best_lr = cv_model_lr.bestModel.stages[-1]

lr_final = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=best_lr.getMaxIter(),
    regParam=best_lr.getRegParam(),
    elasticNetParam=best_lr.getElasticNetParam()
)

stages_lr_final = [s for s in stages if not isinstance(s, LogisticRegression)]
stages_lr_final.append(lr_final)

pipeline_lr_final = Pipeline(stages=stages_lr_final)
model_lr_final = pipeline_lr_final.fit(train_data_weighted)
predictions_lr_final = model_lr_final.transform(test_data)

In [57]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_lr_final)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_lr_final)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_lr_final)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_lr_final)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_lr_final)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9305
2. Accuracy:      0.9222
3. Precision:     0.9231
4. Recall:        0.9222
5. F1-Score:      0.9226


In [ ]:
y_compare = predictions_lr_final.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare["label"], y_compare["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.95      0.95    940259
       Delay       0.77      0.80      0.79    204619

    accuracy                           0.92   1144878
   macro avg       0.87      0.87      0.87   1144878
weighted avg       0.92      0.92      0.92   1144878



In [59]:
lr_model = model_lr_final.stages[-1]
coefficients = lr_model.coefficients.toArray()

features_metadata = predictions_lr_final.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

feature_imp_df = feature_imp_df.sort_values(by='Absolute_Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===
                   Feature_Name  Coefficient  Absolute_Importance
0                AIRLINE_Vec_DL    -0.275806             0.275806
1                AIRLINE_Vec_NK     0.185455             0.185455
2                AIRLINE_Vec_WN    -0.184140             0.184140
3                  MONTH_Vec_10    -0.180882             0.180882
4                AIRLINE_Vec_F9     0.162682             0.162682
5   DESTINATION_AIRPORT_Vec_LAX     0.150434             0.150434
6   DESTINATION_AIRPORT_Vec_LGA     0.149661             0.149661
7                   MONTH_Vec_2     0.134780             0.134780
8                AIRLINE_Vec_UA    -0.128986             0.128986
9                   MONTH_Vec_9    -0.122438             0.122438
10  DESTINATION_AIRPORT_Vec_ORD     0.073626             0.073626
11       ORIGIN_AIRPORT_Vec_LGA     0.073023             0.073023
12              DEPARTURE_DELAY     0.069579             0.069579
13             

In [60]:
best_gbt = cv_model_gbt.bestModel.stages[-1]

gbt_final = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=best_gbt.getMaxIter(),
    maxDepth=best_gbt.getMaxDepth(),
    stepSize=best_gbt.getStepSize()
)

stages_gbt_final = [s for s in stages_gbt if not isinstance(s, GBTClassifier)]
stages_gbt_final.append(gbt_final)

pipeline_gbt_final = Pipeline(stages=stages_gbt_final)
model_gbt_final = pipeline_gbt_final.fit(train_data_weighted)
predictions_gbt_final = model_gbt_final.transform(test_data)

In [61]:
# ── OPTIMIZATION: Explain execution plan ───────────────────────
print("=== EXECUTION PLAN (GBT predictions) ===")
predictions_gbt_final.explain(extended=True)

=== EXECUTION PLAN (GBT predictions) ===
== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(prediction, UDF('rawPrediction) AS prediction#510303, None)]
+- Project [YEAR#100, MONTH#101, DAY#102, DAY_OF_WEEK#103, AIRLINE#104, FLIGHT_NUMBER#105, TAIL_NUMBER#106, ORIGIN_AIRPORT#107, DESTINATION_AIRPORT#108, SCHEDULED_DEPARTURE#257, DEPARTURE_TIME#258, DEPARTURE_DELAY#111, TAXI_OUT#112, WHEELS_OFF#113, SCHEDULED_TIME#114, ELAPSED_TIME#115, AIR_TIME#116, DISTANCE#117, WHEELS_ON#118, TAXI_IN#119, SCHEDULED_ARRIVAL#259, ARRIVAL_TIME#260, ARRIVAL_DELAY#122, DIVERTED#123, CANCELLED#124, ... 21 more fields]
   +- Project [YEAR#100, MONTH#101, DAY#102, DAY_OF_WEEK#103, AIRLINE#104, FLIGHT_NUMBER#105, TAIL_NUMBER#106, ORIGIN_AIRPORT#107, DESTINATION_AIRPORT#108, SCHEDULED_DEPARTURE#257, DEPARTURE_TIME#258, DEPARTURE_DELAY#111, TAXI_OUT#112, WHEELS_OFF#113, SCHEDULED_TIME#114, ELAPSED_TIME#115, AIR_TIME#116, DISTANCE#117, WHEELS_ON#118, TAXI_IN#119, SCHEDULED_ARRIVAL#259, ARRIVAL_TIME#26

In [62]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_gbt_final)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_gbt_final)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_gbt_final)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_gbt_final)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_gbt_final)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9357
2. Accuracy:      0.9088
3. Precision:     0.9159
4. Recall:        0.9088
5. F1-Score:      0.9114


In [63]:
y_compare_gbt = predictions_gbt.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_gbt["label"], y_compare_gbt["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.92      0.94    940259
       Delay       0.69      0.83      0.75    204619

    accuracy                           0.90   1144878
   macro avg       0.83      0.88      0.85   1144878
weighted avg       0.91      0.90      0.91   1144878



In [64]:
gbt_model = model_gbt_final.stages[-1]
feature_importances = gbt_model.featureImportances.toArray()

features_metadata = predictions_gbt_final.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (GBT) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (GBT) ===
                   Feature_Name  Importance
0               DEPARTURE_DELAY    0.911564
1                      DISTANCE    0.011480
2                AIRLINE_Vec_WN    0.008970
3                AIRLINE_Vec_UA    0.006499
4   DESTINATION_AIRPORT_Vec_LAX    0.004910
5   DESTINATION_AIRPORT_Vec_ORD    0.004841
6                   MONTH_Vec_2    0.004253
7                AIRLINE_Vec_DL    0.003940
8        ORIGIN_AIRPORT_Vec_LGA    0.003647
9                AIRLINE_Vec_HA    0.003506
10  DESTINATION_AIRPORT_Vec_LGA    0.003375
11                  MONTH_Vec_1    0.003215
12             DEP_TIME_MINUTES    0.003040
13               AIRLINE_Vec_US    0.002119
14       ORIGIN_AIRPORT_Vec_ATL    0.001933


In [65]:
# ── Unpersist after training ─────────────────────────────────────
flights.unpersist()
train_data.unpersist()
test_data.unpersist()
train_data_weighted.unpersist()
print("Unpersisted all cached data.")

Unpersisted all cached data.


In [66]:
total_time = time.time() - start_notebook
print(f"TOTAL TIME: {total_time:.2f} s")

TOTAL TIME: 3755.62 s
